# Pandas Notebook 03: Missing values and data cleaning

This notebook introduces practical data cleaning in pandas. We will work with a small messy dataset and practice finding missing values, fixing text, removing duplicates, correcting data types, and building a simple cleaning workflow.

## Notebook objectives

By the end of this notebook, you should be able to:

- explain why data cleaning matters before analysis
- detect missing values with `isna()`, `notna()`, and `sum()`
- compare different strategies for handling missing data
- remove duplicate rows
- clean text columns with `strip()`, `lower()`, and `replace()`
- fix data types with `astype()`
- parse date strings with `pd.to_datetime()`
- combine several steps into a small cleaning workflow

## 1. Why data cleaning matters

Real-world data is often messy. You may find:

- missing values
- duplicated rows
- inconsistent text such as `" Bogota"`, `"BOGOTA"`, and `"bogotá"`
- numbers stored as text
- invalid date strings

If we do not clean data carefully, our summaries, charts, and models can be misleading.

There is also an important idea to remember: the **best** cleaning choice depends on the context. Sometimes it is reasonable to drop missing rows. In other cases, that would throw away too much important information. Good cleaning is about making thoughtful decisions, not only using the right syntax.

In [2]:
import pandas as pd

## A messy dataset for practice

We will create a small customer dataset inspired by a messy spreadsheet export. It includes:

- missing numeric values
- missing categorical values
- duplicated rows
- inconsistent text formatting
- date strings in mixed formats

In [3]:
messy_df = pd.DataFrame(
    {
        "customer_id": ["101", "102", "103", "104", "104", "105", " 106 ", "107"],
        "name": [" Ana ", "LUIS", "marta", "John  ", "John  ", None, "Sofia", " Camilo"],
        "city": ["Bogota", " bogota ", "MEDELLIN", None, None, "Cali", "CALI ", " Bogotá "],
        "segment": ["Retail", " retail", None, "Corporate", "Corporate", "Home Office", "RETAIL", "Retail "],
        "monthly_spend": [120.5, None, 300.0, 450.0, 450.0, None, 80.0, 210.0],
        "orders_last_month": ["3", "1", "5", "2", "2", "0", "1", "4"],
        "signup_date": ["2024-01-05", "2024/02/10", "March 3, 2024", "2024-04-01", "2024-04-01", None, "2024-13-01", "2024-05-22"],
        "active": ["Yes", "No", "Yes", "Yes", "Yes", "No", "Yes", None],
        "referral_code": [None, None, "FRIEND10", None, None, None, None, None],
    }
)

messy_df

,customer_id,name,city,segment,monthly_spend,orders_last_month,signup_date,active,referral_code
0,101,Ana,Bogota,Retail,120.5,3,2024-01-05,Yes,NaN
1,102,LUIS,bogota,retail,NaN,1,2024/02/10,No,NaN
2,103,marta,MEDELLIN,NaN,300.0,5,"March 3, 2024",Yes,FRIEND10
3,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
4,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
5,105,NaN,Cali,Home Office,NaN,0,NaN,No,NaN
6,106,Sofia,CALI,RETAIL,80.0,1,2024-13-01,Yes,NaN
7,107,Camilo,Bogotá,Retail,210.0,4,2024-05-22,NaN,NaN


## 2. Detecting missing values with `isna()`, `notna()`, and `sum()`

The first step in cleaning is understanding where the missing data is.

- `isna()` shows `True` where values are missing
- `notna()` shows `True` where values are present
- `sum()` helps count how many `True` values appear in each column

In [4]:
messy_df

,customer_id,name,city,segment,monthly_spend,orders_last_month,signup_date,active,referral_code
0,101,Ana,Bogota,Retail,120.5,3,2024-01-05,Yes,NaN
1,102,LUIS,bogota,retail,NaN,1,2024/02/10,No,NaN
2,103,marta,MEDELLIN,NaN,300.0,5,"March 3, 2024",Yes,FRIEND10
3,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
4,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
5,105,NaN,Cali,Home Office,NaN,0,NaN,No,NaN
6,106,Sofia,CALI,RETAIL,80.0,1,2024-13-01,Yes,NaN
7,107,Camilo,Bogotá,Retail,210.0,4,2024-05-22,NaN,NaN


In [5]:
messy_df.isna().sum()

customer_id          0
name                 1
city                 2
segment              1
monthly_spend        2
orders_last_month    0
signup_date          1
active               1
referral_code        7
dtype: int64

In [6]:
print("Missing values by column:\n")
print(messy_df.isna().sum())
print("\nNon-missing values by column:\n")
print(messy_df.notna().sum())

Missing values by column:

customer_id          0
name                 1
city                 2
segment              1
monthly_spend        2
orders_last_month    0
signup_date          1
active               1
referral_code        7
dtype: int64

Non-missing values by column:

customer_id          8
name                 7
city                 6
segment              7
monthly_spend        6
orders_last_month    8
signup_date          7
active               7
referral_code        1
dtype: int64


## 3. Understanding missingness by column

Counting missing values is helpful, but percentages can be even more useful. A column with 1 missing value out of 1,000 rows is different from a column with 1 missing value out of 3 rows.

Looking at missingness by column helps us ask better questions:

- Is the column still useful?
- Are too many rows incomplete?
- Is a column missing so often that we should drop it?

In [7]:
missing_summary = pd.DataFrame(
    {
        "missing_count": messy_df.isna().sum(),
        "missing_pct": ((messy_df.isna().sum() / len(messy_df)) * 100),
    }
)

missing_summary.sort_values("missing_count", ascending=False)

,missing_count,missing_pct
referral_code,7,87.5
city,2,25.0
monthly_spend,2,25.0
name,1,12.5
segment,1,12.5
signup_date,1,12.5
active,1,12.5
customer_id,0,0.0
orders_last_month,0,0.0


## 4. Strategies for handling missing values

There is no single best strategy for missing data. The right choice depends on the business question, the size of the dataset, and how important the missing column is.

Common strategies include:

- dropping rows
- dropping columns
- filling with a constant such as `"unknown"` or `0`
- filling numeric values with the mean or median
- filling categorical values with the mode

Each strategy has tradeoffs. Dropping rows is simple, but you may lose a lot of data. Filling values keeps more rows, but it adds assumptions.

In [8]:
# Drop rows where monthly_spend is missing.
drop_rows_df = messy_df.dropna(subset=["monthly_spend"])

print("Original shape:", messy_df.shape)
print("After dropping rows with missing monthly_spend:", drop_rows_df.shape)
drop_rows_df

Original shape: (8, 9)
After dropping rows with missing monthly_spend: (6, 9)


,customer_id,name,city,segment,monthly_spend,orders_last_month,signup_date,active,referral_code
0,101,Ana,Bogota,Retail,120.5,3,2024-01-05,Yes,NaN
2,103,marta,MEDELLIN,NaN,300.0,5,"March 3, 2024",Yes,FRIEND10
3,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
4,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
6,106,Sofia,CALI,RETAIL,80.0,1,2024-13-01,Yes,NaN
7,107,Camilo,Bogotá,Retail,210.0,4,2024-05-22,NaN,NaN


In [9]:
# Drop a column if it has too little useful information.
drop_columns_df = messy_df.drop(columns=["referral_code"])

drop_columns_df.head()

,customer_id,name,city,segment,monthly_spend,orders_last_month,signup_date,active
0,101,Ana,Bogota,Retail,120.5,3,2024-01-05,Yes
1,102,LUIS,bogota,retail,NaN,1,2024/02/10,No
2,103,marta,MEDELLIN,NaN,300.0,5,"March 3, 2024",Yes
3,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes
4,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes


In [10]:
messy_df

,customer_id,name,city,segment,monthly_spend,orders_last_month,signup_date,active,referral_code
0,101,Ana,Bogota,Retail,120.5,3,2024-01-05,Yes,NaN
1,102,LUIS,bogota,retail,NaN,1,2024/02/10,No,NaN
2,103,marta,MEDELLIN,NaN,300.0,5,"March 3, 2024",Yes,FRIEND10
3,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
4,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
5,105,NaN,Cali,Home Office,NaN,0,NaN,No,NaN
6,106,Sofia,CALI,RETAIL,80.0,1,2024-13-01,Yes,NaN
7,107,Camilo,Bogotá,Retail,210.0,4,2024-05-22,NaN,NaN


In [11]:
# Filling with constants is common for categorical columns.
filled_constants_df = messy_df.fillna(
    {
        "name": "unknown",
        "city": "unknown",
        "segment": "unknown",
        "active": "No",
    }
)

filled_constants_df

,customer_id,name,city,segment,monthly_spend,orders_last_month,signup_date,active,referral_code
0,101,Ana,Bogota,Retail,120.5,3,2024-01-05,Yes,NaN
1,102,LUIS,bogota,retail,NaN,1,2024/02/10,No,NaN
2,103,marta,MEDELLIN,unknown,300.0,5,"March 3, 2024",Yes,FRIEND10
3,104,John,unknown,Corporate,450.0,2,2024-04-01,Yes,NaN
4,104,John,unknown,Corporate,450.0,2,2024-04-01,Yes,NaN
5,105,unknown,Cali,Home Office,NaN,0,NaN,No,NaN
6,106,Sofia,CALI,RETAIL,80.0,1,2024-13-01,Yes,NaN
7,107,Camilo,Bogotá,Retail,210.0,4,2024-05-22,No,NaN


In [12]:
messy_df

,customer_id,name,city,segment,monthly_spend,orders_last_month,signup_date,active,referral_code
0,101,Ana,Bogota,Retail,120.5,3,2024-01-05,Yes,NaN
1,102,LUIS,bogota,retail,NaN,1,2024/02/10,No,NaN
2,103,marta,MEDELLIN,NaN,300.0,5,"March 3, 2024",Yes,FRIEND10
3,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
4,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
5,105,NaN,Cali,Home Office,NaN,0,NaN,No,NaN
6,106,Sofia,CALI,RETAIL,80.0,1,2024-13-01,Yes,NaN
7,107,Camilo,Bogotá,Retail,210.0,4,2024-05-22,NaN,NaN


In [15]:
fill_demo = messy_df.copy()

fill_demo["monthly_spend_mean"] = fill_demo["monthly_spend"].fillna(fill_demo["monthly_spend"].mean())
fill_demo["monthly_spend_median"] = fill_demo["monthly_spend"].fillna(fill_demo["monthly_spend"].median())
fill_demo["segment_mode"] = fill_demo["segment"].fillna(fill_demo["segment"].mode()[0])

fill_demo

,customer_id,name,city,segment,monthly_spend,orders_last_month,signup_date,active,referral_code,monthly_spend_mean,monthly_spend_median,segment_mode
0,101,Ana,Bogota,Retail,120.5,3,2024-01-05,Yes,NaN,120.500000,120.5,Retail
1,102,LUIS,bogota,retail,NaN,1,2024/02/10,No,NaN,268.416667,255.0,retail
2,103,marta,MEDELLIN,NaN,300.0,5,"March 3, 2024",Yes,FRIEND10,300.000000,300.0,Corporate
3,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN,450.000000,450.0,Corporate
4,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN,450.000000,450.0,Corporate
5,105,NaN,Cali,Home Office,NaN,0,NaN,No,NaN,268.416667,255.0,Home Office
6,106,Sofia,CALI,RETAIL,80.0,1,2024-13-01,Yes,NaN,80.000000,80.0,RETAIL
7,107,Camilo,Bogotá,Retail,210.0,4,2024-05-22,NaN,NaN,210.000000,210.0,Retail


### Choosing between mean, median, and mode

- **Mean** uses the average. It can work well when the numbers are fairly balanced.
- **Median** is often safer when a numeric column has extreme values, because it is less affected by outliers.
- **Mode** is useful for categorical columns because it fills with the most common category.

Again, context matters. You should always think about what the missing values might represent before filling them.

## 5. Removing duplicates

Duplicate rows can happen when data is copied, merged incorrectly, or exported more than once. Duplicates can cause counts and totals to be wrong.

In [26]:
messy_df

,customer_id,name,city,segment,monthly_spend,orders_last_month,signup_date,active,referral_code
0,101,Ana,Bogota,Retail,120.5,3,2024-01-05,Yes,NaN
1,102,LUIS,bogota,retail,NaN,1,2024/02/10,No,NaN
2,103,marta,MEDELLIN,NaN,300.0,5,"March 3, 2024",Yes,FRIEND10
3,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
4,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
5,105,NaN,Cali,Home Office,NaN,0,NaN,No,NaN
6,106,Sofia,CALI,RETAIL,80.0,1,2024-13-01,Yes,NaN
7,107,Camilo,Bogotá,Retail,210.0,4,2024-05-22,NaN,NaN


In [27]:
messy_df.drop_duplicates()

,customer_id,name,city,segment,monthly_spend,orders_last_month,signup_date,active,referral_code
0,101,Ana,Bogota,Retail,120.5,3,2024-01-05,Yes,NaN
1,102,LUIS,bogota,retail,NaN,1,2024/02/10,No,NaN
2,103,marta,MEDELLIN,NaN,300.0,5,"March 3, 2024",Yes,FRIEND10
3,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
5,105,NaN,Cali,Home Office,NaN,0,NaN,No,NaN
6,106,Sofia,CALI,RETAIL,80.0,1,2024-13-01,Yes,NaN
7,107,Camilo,Bogotá,Retail,210.0,4,2024-05-22,NaN,NaN


In [30]:
print("Duplicate rows:\n")
print(messy_df[messy_df.duplicated()])

deduplicated_df = messy_df.drop_duplicates()
print("\nShape after removing duplicates:", deduplicated_df.shape)
deduplicated_df

Duplicate rows:

  customer_id    name city    segment  monthly_spend orders_last_month  \
4         104  John    NaN  Corporate          450.0                 2   

  signup_date active referral_code  
4  2024-04-01    Yes           NaN  

Shape after removing duplicates: (7, 9)


,customer_id,name,city,segment,monthly_spend,orders_last_month,signup_date,active,referral_code
0,101,Ana,Bogota,Retail,120.5,3,2024-01-05,Yes,NaN
1,102,LUIS,bogota,retail,NaN,1,2024/02/10,No,NaN
2,103,marta,MEDELLIN,NaN,300.0,5,"March 3, 2024",Yes,FRIEND10
3,104,John,NaN,Corporate,450.0,2,2024-04-01,Yes,NaN
5,105,NaN,Cali,Home Office,NaN,0,NaN,No,NaN
6,106,Sofia,CALI,RETAIL,80.0,1,2024-13-01,Yes,NaN
7,107,Camilo,Bogotá,Retail,210.0,4,2024-05-22,NaN,NaN


## 6. Cleaning text columns

Text data often needs simple cleanup before analysis.

Useful string methods include:

- `strip()` to remove extra spaces at the start or end
- `lower()` to standardize capitalization
- `replace()` to fix known text variations

In [31]:
text_clean_df = messy_df.copy()

In [35]:
text_clean_df["name"].fillna("unknown").str.strip().str.title()

0        Ana
1       Luis
2      Marta
3       John
4       John
5    Unknown
6      Sofia
7     Camilo
Name: name, dtype: str

In [17]:
text_clean_df = messy_df.copy()

text_clean_df["name"] = text_clean_df["name"].fillna("unknown").str.strip().str.title()
text_clean_df["city"] = (
    text_clean_df["city"]
    .fillna("unknown")
    .str.strip()
    .str.lower()
    .str.replace("bogotá", "bogota", regex=False)
)
text_clean_df["segment"] = (
    text_clean_df["segment"]
    .fillna("unknown")
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

text_clean_df[["name", "city", "segment"]]

,name,city,segment
0,Ana,bogota,retail
1,Luis,bogota,retail
2,Marta,medellin,unknown
3,John,unknown,corporate
4,John,unknown,corporate
5,Unknown,cali,home_office
6,Sofia,cali,retail
7,Camilo,bogota,retail


## 7. Changing data types with `astype()`

Sometimes numbers are stored as text. This can cause problems if you want to sort, calculate averages, or build charts.

After cleaning the text values, we can convert columns to more appropriate types.

In [36]:
messy_df.dtypes

customer_id              str
name                     str
city                     str
segment                  str
monthly_spend        float64
orders_last_month        str
signup_date              str
active                   str
referral_code            str
dtype: object

In [37]:
dtype_fix_df = messy_df.copy()
dtype_fix_df["customer_id"] = dtype_fix_df["customer_id"].str.strip().astype(int)
dtype_fix_df["orders_last_month"] = dtype_fix_df["orders_last_month"].astype(int)

print(dtype_fix_df.dtypes)
print("\nPreview:\n")
dtype_fix_df[["customer_id", "orders_last_month"]].head()

customer_id            int64
name                     str
city                     str
segment                  str
monthly_spend        float64
orders_last_month      int64
signup_date              str
active                   str
referral_code            str
dtype: object

Preview:



,customer_id,orders_last_month
0,101,3
1,102,1
2,103,5
3,104,2
4,104,2


## 8. Parsing dates with `to_datetime`

Dates often arrive as plain text. `pd.to_datetime()` helps convert them into a real datetime type.

Using `errors="coerce"` is helpful because invalid dates become `NaT` instead of crashing the code. `NaT` is pandas' missing value marker for datetime data.

In [38]:
date_fix_df = messy_df.copy()
date_fix_df["signup_date_parsed"] = pd.to_datetime(date_fix_df["signup_date"], errors="coerce")

print(date_fix_df[["signup_date", "signup_date_parsed"]])
print("\nParsed dtype:", date_fix_df["signup_date_parsed"].dtype)

     signup_date signup_date_parsed
0     2024-01-05         2024-01-05
1     2024/02/10                NaT
2  March 3, 2024                NaT
3     2024-04-01         2024-04-01
4     2024-04-01         2024-04-01
5            NaN                NaT
6     2024-13-01                NaT
7     2024-05-22         2024-05-22

Parsed dtype: datetime64[us]


## 9. Building a simple cleaning workflow

In real projects, we usually combine several cleaning steps into one repeatable workflow.

Below is a small example. It is not the only correct solution, but it shows a reasonable path from messy data to analysis-ready data.

In [39]:
cleaned_df = messy_df.drop_duplicates().copy()

# Clean text columns.
cleaned_df["name"] = cleaned_df["name"].fillna("unknown").str.strip().str.title()
cleaned_df["city"] = (
    cleaned_df["city"]
    .fillna("unknown")
    .str.strip()
    .str.lower()
    .str.replace("bogotá", "bogota", regex=False)
)
cleaned_df["segment"] = (
    cleaned_df["segment"]
    .fillna("unknown")
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
cleaned_df["active"] = cleaned_df["active"].fillna("No").str.strip().str.lower()

# Fill numeric missing values.
cleaned_df["monthly_spend"] = cleaned_df["monthly_spend"].fillna(cleaned_df["monthly_spend"].median())

# Fix dtypes.
cleaned_df["customer_id"] = cleaned_df["customer_id"].str.strip().astype(int)
cleaned_df["orders_last_month"] = cleaned_df["orders_last_month"].astype(int)
cleaned_df["signup_date"] = pd.to_datetime(cleaned_df["signup_date"], errors="coerce")

# Drop a mostly empty column.
cleaned_df = cleaned_df.drop(columns=["referral_code"])

cleaned_df

,customer_id,name,city,segment,monthly_spend,orders_last_month,signup_date,active
0,101,Ana,bogota,retail,120.5,3,2024-01-05,yes
1,102,Luis,bogota,retail,210.0,1,NaT,no
2,103,Marta,medellin,unknown,300.0,5,NaT,yes
3,104,John,unknown,corporate,450.0,2,2024-04-01,yes
5,105,Unknown,cali,home_office,210.0,0,NaT,no
6,106,Sofia,cali,retail,80.0,1,NaT,yes
7,107,Camilo,bogota,retail,210.0,4,2024-05-22,no


In [40]:
print("Remaining missing values after cleaning:\n")
print(cleaned_df.isna().sum())
print("\nData types after cleaning:\n")
print(cleaned_df.dtypes)

Remaining missing values after cleaning:

customer_id          0
name                 0
city                 0
segment              0
monthly_spend        0
orders_last_month    0
signup_date          4
active               0
dtype: int64

Data types after cleaning:

customer_id                   int64
name                            str
city                            str
segment                         str
monthly_spend               float64
orders_last_month             int64
signup_date          datetime64[us]
active                          str
dtype: object


## Exercises

Try these short exercises with `messy_df` or `cleaned_df`.

1. Count the missing values in each column of `messy_df`.
2. Create a new DataFrame without duplicate rows.
3. Fill missing values in `monthly_spend` using the median.
4. Clean the `city` column so values like `" bogota "`, `"Bogotá"`, and `"BOGOTA"` become consistent.
5. Convert `signup_date` to a datetime column and identify which rows became `NaT`.
6. Choose one column and explain which cleaning strategy you think is best for it, and why.

In [ ]:
# Write your exercise answers here.

## Mini challenge

Here is a second messy dataset. Your goal is to make it analysis-ready.

In [41]:
challenge_df = pd.DataFrame(
    {
        "order_id": ["201", "202", "202", "203", "204", "205"],
        "customer_name": [" Ana", "LUIS ", "LUIS ", "Marta", None, "sofia"],
        "category": ["Books", " books", " books", None, "Electronics", "electronics"],
        "amount": [15.0, None, None, 22.0, 199.99, 45.0],
        "order_date": ["2024-06-01", "2024/06/02", "2024/06/02", "June 5 2024", "not a date", None],
        "city": ["Bogota", " bogotá", " bogotá", "Cali ", "MEDELLIN", None],
        "coupon_code": [None, None, None, "SAVE5", None, None],
    }
)

challenge_df

,order_id,customer_name,category,amount,order_date,city,coupon_code
0,201,Ana,Books,15.00,2024-06-01,Bogota,NaN
1,202,LUIS,books,NaN,2024/06/02,bogotá,NaN
2,202,LUIS,books,NaN,2024/06/02,bogotá,NaN
3,203,Marta,NaN,22.00,June 5 2024,Cali,SAVE5
4,204,NaN,Electronics,199.99,not a date,MEDELLIN,NaN
5,205,sofia,electronics,45.00,NaN,NaN,NaN


### Your task

Clean `challenge_df` so it is ready for analysis.

Try to do at least these steps:

1. Remove duplicate rows.
2. Standardize text in `customer_name`, `category`, and `city`.
3. Fill missing values in a reasonable way.
4. Convert `order_id` to an integer type.
5. Parse `order_date` with `pd.to_datetime()`.
6. Decide whether to keep or drop `coupon_code`, and explain why.

After cleaning, answer these questions:

- How many valid rows are left?
- Which orders have `amount > 20`?
- Which rows still have missing values, if any?

In [ ]:
# Write your mini challenge solution here.

## Cleaning checklist

When you start cleaning a dataset, it helps to work through a simple checklist:

- inspect the dataset and identify obvious problems
- count missing values by column
- decide whether to drop or fill missing values
- check for duplicate rows
- standardize text columns
- verify data types for numbers, categories, and dates
- recheck missing values and data types after cleaning
- save or reuse your cleaning steps so the workflow is repeatable

## Key takeaways

- data cleaning matters because messy data can lead to wrong conclusions
- `isna()`, `notna()`, and `sum()` are core tools for understanding missing data
- dropping, filling, and keeping missing values all involve tradeoffs
- duplicate rows can distort counts and summaries
- text cleaning often starts with `strip()`, `lower()`, and `replace()`
- `astype()` and `pd.to_datetime()` help turn text into useful analysis-ready types
- the best cleaning workflow depends on the context and the meaning of the data